In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Nạp API key từ file .env
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

# Khởi tạo Client với SDK mới
client = genai.Client(api_key=api_key)

try:
    # Sử dụng model gemini-3.8-flash theo khuyến nghị của Google
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents="Nói 'Xin chào!' bằng tiếng anh.",

    )
    print("Kết quả:", response.text)
except Exception as e:
    print("Lỗi khi gọi API:", e)

Kết quả: Bạn có thể nói là: **"Hello!"** hoặc **"Hi!"** nhé.


In [5]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Hãy tóm tắt tác dụng của Docker trong 2 gạch đầu dòng.",
    config=types.GenerateContentConfig(
        temperature=0.2,          # Logic, chính xác, ít lan man 0.0 nghiem tuc => 1.0 bay bong 
        max_output_tokens=100,    # Giới hạn độ dài để tiết kiệm hạn mức token
    )
)

print(response.text)

Dưới đây là tóm tắt tác dụng của Docker trong 2 gạch đầu dòng:

* **Đảm bảo tính nhất quán:** Đóng gói ứng dụng cùng toàn bộ môi trường (thư viện, cấu hình) vào các "container", giúp ứng dụng chạy ổn định trên mọi máy tính mà không gặp lỗi "trên máy tôi thì chạy được".
* **Tối ưu hóa triển khai:** Giúp việc cài đặt, di chuyển và mở rộng


Markdown
# Ghi nhớ: Kỹ thuật Chain-of-Thought (CoT) Prompting

## 1. Bản chất & Cơ chế
* **Định nghĩa:** Bắt LLM trình bày từng bước suy luận trung gian (dàn giáo suy nghĩ) trước khi chốt đáp án cuối cùng, thay vì nhảy cóc ra kết quả ngay.
* **Cơ chế:** LLM sinh chữ bằng cách dự đoán token tiếp theo. Khi viết ra bước 1, chính bước 1 trở thành ngữ cảnh đầu vào giúp mô hình tính toán bước 2 chuẩn xác hơn.
* **Bản chất:** Không nạp thêm kiến thức mới, chỉ "mở khóa" khả năng suy luận logic sẵn có của mô hình.

---

## 2. Hai hình thức triển khai

| Tiêu chí | Zero-shot CoT | Few-shot CoT |
| :--- | :--- | :--- |
| **Bản chất** | Dùng câu "thần chú", không cần ví dụ | Đưa kèm 1–2 bài toán mẫu đã giải sẵn |
| **Cách dùng** | Thêm: *"Hãy suy nghĩ từng bước một rồi đưa ra đáp án."* (*"Let's think step by step."*) | Đưa cặp mẫu `(User: bài toán -> Assistant: các bước giải + đáp án)` trước câu hỏi thật |
| **Ưu điểm** | Nhanh, ngắn gọn, tiết kiệm token | Ép khuôn định dạng và phong cách suy luận chính xác theo ý muốn |
| **Khi nào dùng**| Bài toán logic phổ thông, kiểm tra nhanh | Bài toán đặc thù, cấu trúc phức tạp, cần chuẩn hóa output |

In [6]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Nạp API key từ file .env
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

client = genai.Client(api_key=api_key)

# 2. Đề bài toán logic nhiều bước
bai_toan = (
    "Một lớp có 25 học sinh. 3/5 số đó là nữ. "
    "Trong số nữ, 4 bạn đeo kính. "
    "Hỏi có bao nhiêu bạn nữ KHÔNG đeo kính?"
)

# 3. Kỹ thuật Zero-shot CoT: Ghép câu 'thần chú' vào cuối đề bài
prompt_cot = f"{bai_toan}\n\nHãy suy nghĩ từng bước một, rồi đưa ra đáp án cuối cùng."

# 4. Cấu hình system instruction và temperature=0 cho logic ổn định
config = types.GenerateContentConfig(
    system_instruction="Bạn là trợ giảng Toán, luôn trình bày lời giải cẩn thận và rõ ràng từng bước.",
    temperature=0.0,         # Đặt 0 để suy luận logic ổn định, tránh bịa số
    max_output_tokens=600,   # Giới hạn token để tiết kiệm quota
)

try:
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",  # Hoặc "gemini-3.8-flash"
        contents=prompt_cot,
        config=config,
    )
    print("--- LỜI GIẢI CỦA GEMINI ---")
    print(response.text)
except Exception as e:
    print("Lỗi:", e)

--- LỜI GIẢI CỦA GEMINI ---
Chào bạn, mình là trợ giảng Toán của bạn. Chúng ta sẽ cùng giải bài toán này theo từng bước nhé:

**Bước 1: Tính số học sinh nữ trong lớp.**
Theo đề bài, lớp có 25 học sinh và số học sinh nữ chiếm $\frac{3}{5}$ tổng số học sinh.
Số học sinh nữ là:
$25 \times \frac{3}{5} = 15$ (học sinh)

**Bước 2: Tính số học sinh nữ không đeo kính.**
Chúng ta biết rằng trong số 15 bạn nữ này, có 4 bạn đeo kính. Để tìm số bạn nữ không đeo kính, ta lấy tổng số bạn nữ trừ đi số bạn nữ có đeo kính:
$15 - 4 = 11$ (học sinh)

**Đáp số:** Có **11** bạn nữ không đeo kính.


In [7]:
prompt_few_shot = """Bạn là trợ lý giải toán logic, luôn phân tích theo các bước đánh số rõ ràng.

Ví dụ mẫu:
Câu hỏi: Có 12 quả táo, chia đều vào 3 rổ, sau đó bỏ bớt 1 rổ đi. Hỏi còn lại bao nhiêu quả táo?
Lời giải:
Bước 1: 12 chia 3 = 4 quả mỗi rổ.
Bước 2: Bỏ bớt 1 rổ tức là bớt đi 4 quả.
Bước 3: Số táo còn lại là 12 - 4 = 8 quả.
Đáp án: 8 quả.

---
Bây giờ hãy giải bài toán sau:
Câu hỏi: Một cửa hàng giảm giá 20%, sau đó giảm tiếp 10% trên giá đã giảm. Hỏi tổng cộng giảm bao nhiêu phần trăm so với giá gốc?
Lời giải:
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt_few_shot,
    config=types.GenerateContentConfig(temperature=0.0),
)

print(response.text)

Lời giải:

Bước 1: Giả sử giá gốc của sản phẩm là 100%. Sau khi giảm giá 20%, giá của sản phẩm còn lại là: 100% - 20% = 80% (giá gốc).

Bước 2: Tính mức giảm tiếp theo 10% trên giá đã giảm (80%): 10% × 80% = 8%.

Bước 3: Giá cuối cùng của sản phẩm là: 80% - 8% = 72% (giá gốc).

Bước 4: Tổng cộng phần trăm đã giảm so với giá gốc là: 100% - 72% = 28%.

Đáp án: 28%.
